# OntoNotes v5.0 for NER

## Introduction

**Objective**: To explore `[OntoNotes]` dataset for NER task and convert it to `CONLL` format.


- Project: [https://catalog.ldc.upenn.edu/LDC2013T19](https://catalog.ldc.upenn.edu/LDC2013T19)
- Release Year: 2013
- Data Sources: telephone talks, newswire, newsgroups, weblogs, religious text, etc.
- Format: Penn Treebank
- Languages: English, Chinese, Arabic 
- Usage: NER, POS, Coreference resolution

Following steps that we will take into action for achiving our objective.
- load this dataset from huggingface itself.
- convert to conll format and save to disk. 

At the end we will have train.conll, test.conll and validate.conll files.

## Implementation

### Import libraries

Importing just huggingface's `datasets` library.

In [1]:
import datasets

ModuleNotFoundError: No module named 'datasets'

Assert the library version to make sure we are using that as expected versions. 

In [ ]:
assert datasets.__version__ == '2.3.2'

### Load dataset

Loading a well prepared `Ontonotes v5` dataset from huggingface itself.

In [ ]:
dset = datasets.load_dataset("conll2012_ontonotesv5", "english_v4")

let's see how many samples which we have ? A dataset is already splitted out into train/test/validate. 

We have `1940` samples in training and `222` for testing.
 
Here `Sample` mean `Document` and single document does have multiple sentences.

In [ ]:
dset

In [ ]:
train_set = dset['train']

In [ ]:
print(train_set)

### Dataset understanding

This dataset is being used for multiple purpose. like NER (Named Entity recognization), POS (Part of speech tagging) and Coreference resolution.

Let have look for features, here dataset have words, pos_tags, parse_tree, word_senses, named_entities featues.

In [ ]:
train_set.features

As we know that a dataset made of document and single document has multiplt sentences. An each sentences has one feature called `named_entities` inside it sub featues called `names`. that contain the actual entity name.

In [ ]:
train_set.features['sentences'][0]['named_entities'].feature

In [ ]:
names = train_set.features['sentences'][0]['named_entities'].feature.names
print(names)

A document sample composed of "document_id" and it's sentences. 

In [ ]:
train_set[0]['document_id']

In [ ]:
doc_senteces = train_set[0]['sentences']
len(doc_senteces)

In [ ]:
doc_senteces[0]

Our goal is to do NER so we will be using only `words` and `named_entities` featues. 

In [ ]:
print(doc_senteces[1]['words'])

In [ ]:
entity_names = doc_senteces[1]['named_entities']
print(entity_names)

decoded_entity_names = [names[i] for i in entity_names]
print(decoded_entity_names)

### Convert to conll

A `save_as_conll` function is used to write dataset into file as conll format. This function will iterate every document from fullset and write pair of word and ner into file.

Format:
```
WORD \t TAG
``` 

In [ ]:
import tqdm

def save_as_conll(data_set, out_path):
    total_sentences = 0
    with open(out_path, mode='w', encoding='utf-8') as _file:
        for i, _doc in enumerate(tqdm.tqdm(data_set)):
            for _s in _doc['sentences']:
                total_sentences += 1
                for _w, _t in zip(_s['words'], _s['named_entities']):
                    _file.write(f"{_w}\t{names[_t]}\n")
                _file.write(f"\n\n")
    print("Total_sentences:", total_sentences)

In [ ]:
save_as_conll(train_set, "data/ontonotes/train.conll")

In [ ]:
save_as_conll(dset['test'], "data/ontonotes/test.conll")

In [ ]:
save_as_conll(dset['validation'], "data/ontonotes/validation.conll")

Now we have ontonotes 5.0 dataset in conll format. which has around `70K` of training samples and `10K` samples for testing.

In [ ]:
!du -k data/ontonotes/*

### Summary

| Split Name | # Documents | # Sentences | # Disk occupy |
| --- | --- | --- | --- |
| Train | 1940 | 75187 | 10052 KB |
| Test | 222 | 9479 | 1324 KB |
| Validate | 222 | 9603 | 1272 KB |

## Reference
- [https://huggingface.co/datasets/conll2012_ontonotesv5](https://huggingface.co/datasets/conll2012_ontonotesv5)
- [https://huggingface.co/docs/datasets/loading](https://huggingface.co/docs/datasets/loading)
- [https://huggingface.co/docs/datasets/access](https://huggingface.co/docs/datasets/access)
- [https://huggingface.co/docs/datasets/v2.3.2/en/package_reference/main_classes](https://huggingface.co/docs/datasets/v2.3.2/en/package_reference/main_classes)